In [1]:
# 0. bufer para todas las capas
import os
import geopandas as gpd

"C:/Geodata/Analisis datos a raster/"
# Ruta de entrada (carpeta con archivos .shp)
carpeta_entrada = r'C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\capas_historicas'

# Ruta de salida (donde se guardarán los nuevos shapefiles)
carpeta_salida = r'C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\capas_historicas_buffer'

# Crear la carpeta de salida si no existe
os.makedirs(carpeta_salida, exist_ok=True)

# Iterar sobre todos los archivos .shp en la carpeta de entrada
for archivo in os.listdir(carpeta_entrada):
    if archivo.endswith('.shp'):
        ruta_entrada = os.path.join(carpeta_entrada, archivo)

        # Leer el shapefile
        gdf = gpd.read_file(ruta_entrada)

        # Aplicar buffer negativo de -5 (unidades dependen del CRS)
        gdf_buffer = gdf.copy()
        gdf_buffer['geometry'] = gdf_buffer['geometry'].buffer(-2.5)

        # Eliminar geometrías vacías o inválidas después del buffer
        gdf_buffer = gdf_buffer[gdf_buffer.is_valid & ~gdf_buffer.is_empty]

        # Nombre del nuevo archivo
        ruta_salida = os.path.join(carpeta_salida, archivo)

        # Guardar el shapefile con buffer aplicado
        gdf_buffer.to_file(ruta_salida)

        print(f"Procesado: {archivo}")


Procesado: 202406.shp
Procesado: 202407.shp
Procesado: 202408.shp
Procesado: 202409.shp
Procesado: 202410.shp
Procesado: 202411.shp
Procesado: 202412.shp
Procesado: 202501.shp
Procesado: 202502.shp
Procesado: 202503.shp
Procesado: 202505.shp
Procesado: 202506.shp
Procesado: 202507.shp
Procesado: 202508.shp
Procesado: 202509.shp
Procesado: 202510.shp
Procesado: 202511.shp
Procesado: 202512.shp
Procesado: 202601.shp
Procesado: 202602.shp
Procesado: 202604.shp
Procesado: 202605.shp
Procesado: 202606.shp
Procesado: 202607.shp
Procesado: 202608.shp
Procesado: 202609.shp


In [ ]:
# 1. SCRIP PERO BUSCA DATOS FALTANTES EN UN RANGO DE FECHA +- 2 MESES Y TOMA EL PRIMER DATO QUE ENCUENTRE
# DURACION DE 12 MINUTOS APROX
import pandas as pd
import geopandas as gpd
import os
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta  # ✅ Mejora menor para manejo de meses exactos

holgura_meses = 6

def obtener_rango_fechas(base_fecha, rango_meses=holgura_meses):
    """Generar un rango de fechas (YYYYMM) dado un rango de meses antes y después."""
    base_fecha = datetime.strptime(base_fecha, "%Y%m")
    fechas = [(base_fecha + relativedelta(months=i)).strftime("%Y%m") for i in range(-rango_meses, rango_meses + 1)]
    return sorted(set(fechas))  # Asegurar que no haya duplicados

def buscar_poligono_en_rango(codsia, rango_fechas, shape_folder):
    """Buscar un polígono específico en las capas dentro de un rango de fechas."""
    for fecha in rango_fechas:
        shape_path = os.path.join(shape_folder, f"{fecha}.shp")
        if os.path.exists(shape_path):
            gdf = gpd.read_file(shape_path)         
            if codsia in gdf['CODSIA'].values:
                gdf_resultado = gdf[gdf['CODSIA'] == codsia]
                gdf_resultado['archivo_origen'] = fecha  # Registrar la fecha del archivo de origen
                return gdf_resultado
    return None

def procesar_cultivos(excel_path, shape_folder, year_filter, output_path):
    # Cargar datos del Excel
    df = pd.read_excel(excel_path, sheet_name = 'Busca_poligonos')
    
    # Filtrar datos del año deseado
    df = df[df['year'] == year_filter]
    
    # Filtrar duplicados por CODSIA con las reglas especificadas
    df = (
        df.sort_values(by=['Area Neta', 'mes'], ascending=[False, False])  # Ordenar por Area Neta descendente y luego por mes descendente
          .drop_duplicates(subset=['CODSIA'], keep='first')               # Mantener el primer registro único por codsia
    )
    
    df.to_excel('C:/Users/jjruiz/OneDrive - Sector Agro/Geodata/Analisis datos a raster/TEMP/Resultadoyear'+str(year_filter)+'.xlsx')
    
    # Crear una lista para almacenar los GeoDataFrames de cada mes
    capas_mensuales = []
    
    for _, row in df.iterrows():
        # Formatear la fecha a la estructura 'YYYYMM' para buscar el shapefile
        fecha_str = f"{row['year']}{str(row['mes']).zfill(2)}"
        shape_path = os.path.join(shape_folder, f"{fecha_str}.shp")
        
        # Buscar el polígono correspondiente al CODSIA
        if os.path.exists(shape_path):
            gdf_mes = gpd.read_file(shape_path)
            gdf_filtrado = gdf_mes[gdf_mes['CODSIA'] == row['CODSIA']]
            if not gdf_filtrado.empty:
                gdf_filtrado['archivo_origen'] = fecha_str
        else:
            gdf_filtrado = None
        
        # Si no se encontró, buscar en el rango de fechas
        if gdf_filtrado is None or gdf_filtrado.empty:
            rango_fechas = obtener_rango_fechas(fecha_str, rango_meses=holgura_meses)
            gdf_filtrado = buscar_poligono_en_rango(row['CODSIA'], rango_fechas, shape_folder)

            # ✅ Verificación espacial con polígonos ya existentes
            if gdf_filtrado is not None and not gdf_filtrado.empty and capas_mensuales:
                capa_existente = gpd.GeoDataFrame(pd.concat(capas_mensuales, ignore_index=True), crs=capas_mensuales[0].crs)
                intersecciones = gpd.sjoin(gdf_filtrado, capa_existente, how='inner', predicate='intersects')
                if not intersecciones.empty:
                    print(f"Polígono de {row['CODSIA']} (recuperado de rango) ya intersecta con otro existente. Se omite.")
                    gdf_filtrado = None  # No se añade

        # Añadir el polígono encontrado (si existe y no fue descartado) a las capas mensuales
        if gdf_filtrado is not None and not gdf_filtrado.empty:
            capas_mensuales.append(gdf_filtrado)
        else:
            print(f"No se encontró el polígono para CODSIA {row['CODSIA']} en {fecha_str} o en el rango de fechas.")
    
    # Unir todas las capas mensuales en una capa general
    if capas_mensuales:
        capa_general = gpd.GeoDataFrame(pd.concat(capas_mensuales, ignore_index=True), crs=capas_mensuales[0].crs)
        # Guardar la capa general en un nuevo shapefile
        
        #capa_general["geometry"] = capa_general.buffer(-5)
        
        capa_general.to_file(output_path, driver='ESRI Shapefile')
        print(f"Capa general para el año {year_filter} creada exitosamente en {output_path}")
    else:
        print("No se encontraron datos para generar la capa general.")

# Parámetros de entrada
shape_folder = "C:/Users/jjruiz/OneDrive - Sector Agro/Geodata/Analisis datos a raster/capas_historicas_buffer/"  # Carpeta con los shapefiles mensuales
year_filter = 2026  # Año que deseas procesar
excel_path = "C:/Users/jjruiz/OneDrive - Sector Agro/Geodata/Analisis datos a raster/Reporte_Biosal_2026.xlsx"  # Ruta al archivo Excel
output_path = "C:/Users/jjruiz/OneDrive - Sector Agro/Geodata/Analisis datos a raster/TEMP/CP_"+str(year_filter)+".shp"  # Ruta donde se guardará la capa general

# Ejecutar el proceso
procesar_cultivos(excel_path, shape_folder, year_filter, output_path)


In [ ]:
# 2. buscar ste y poligono por periodo,
    # primero: buscar en periodo igual sino se encuentra
    # segundo: buscar en 6 periodos antes sino se encuentra
    # tercero: buscar en todos los periodos posteriores

import os
import pandas as pd
import geopandas as gpd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Ruta del archivo Excel con los códigos CODSIA
archivo_codigos = "C:/Users/jjruiz/OneDrive - Sector Agro/Geodata/Analisis datos a raster/"

# Rutas
ruta_excel = archivo_codigos+"HISTORICO_SUERTES.xlsx"  # Cambia a la ruta de tu archivo
hoja_excel = "Historico"       # Cambia si tu hoja tiene otro nombre
columna_periodo = "Periodo"
columna_codsia = "CODSIA"
columna_tenencia = "Tenencia"
carpeta_shapes = archivo_codigos+"capas_historicas_buffer"  # Carpeta con los .shp (ej: 'shapes/201402.shp')

# Leer Excel y ordenar
df_excel = pd.read_excel(ruta_excel, sheet_name=hoja_excel)
df_excel = df_excel.sort_values(by=columna_periodo).reset_index(drop=True)

# Obtener lista de shapefiles y ordenarlos
shapefiles = sorted([f for f in os.listdir(carpeta_shapes) if f.endswith('.shp')])
shape_periodos = sorted([os.path.splitext(f)[0] for f in shapefiles])  # Solo nombres sin extensión

# Cargar shapefiles en memoria una sola vez
shapes_dict = {}
for shp in shapefiles:
    try:
        capa = gpd.read_file(os.path.join(carpeta_shapes, shp))
        shapes_dict[shp] = capa
    except Exception as e:
        print(f"Error al leer {shp}: {e}")

# Función para generar lista de periodos YYYYMM desde una fecha
def generar_periodos_anteriores_y_posteriores(periodo_str, meses_anteriores=6):
    fecha = datetime.strptime(str(periodo_str), "%Y%m")
    anteriores = [(fecha - relativedelta(months=i)).strftime("%Y%m") for i in range(1, meses_anteriores + 1)]
    posteriores = sorted([p for p in shape_periodos if p >= str(periodo_str)])
    return anteriores, posteriores

# Resultado
resultados = []

# Búsqueda principal
for _, fila in df_excel.iterrows():
    periodo = str(fila[columna_periodo])
    codsia = str(fila[columna_codsia])
    tenencia = str(fila[columna_tenencia])

    anteriores, posteriores = generar_periodos_anteriores_y_posteriores(periodo)

    encontrado = False

    # Buscar en los 6 meses anteriores
    for ant in anteriores:
        shp_nombre = f"{ant}.shp"
        if shp_nombre in shapes_dict:
            capa = shapes_dict[shp_nombre]
            if columna_codsia in capa.columns:
                if codsia in capa[columna_codsia].astype(str).values:
                    resultados.append({
                        "PERIODO": periodo,
                        "CODSIA": codsia,
                        "TENENCIA": tenencia,
                        "CAPA": shp_nombre
                    })
                    encontrado = True
                    break

    # Si no se encontró en los anteriores, buscar hacia adelante
    if not encontrado:
        for post in posteriores:
            shp_nombre = f"{post}.shp"
            if shp_nombre in shapes_dict:
                capa = shapes_dict[shp_nombre]
                if columna_codsia in capa.columns:
                    if codsia in capa[columna_codsia].astype(str).values:
                        resultados.append({
                            "PERIODO": periodo,
                            "CODSIA": codsia,
                            "TENENCIA": tenencia,
                            "CAPA": shp_nombre
                        })
                        encontrado = True
                        break

    # Si no se encuentra en ninguna parte
    if not encontrado:
        resultados.append({
            "PERIODO": periodo,
            "CODSIA": codsia,
            "TENENCIA": tenencia,
            "CAPA": "NO ENCONTRADO"
        })

# Guardar resultado
df_resultados = pd.DataFrame(resultados)
df_resultados.to_excel("informe_resultado_2025.xlsx", index=False)

print("✅ Proceso finalizado. Resultado guardado en: informe_resultado.xlsx")


✅ Proceso finalizado. Resultado guardado en: informe_resultado.xlsx


In [4]:
# 3. Depuracion y ajuste de poligonos, por solape duplicados u ocupando el mismo espacio
'''
OBLIGATORIO CORRER ESTE SCRIPT AL FINAL


En medio del proceso se hace un buffer de -2 centimetros para eliminar contactos de
bordes que generen errores y ayude a identificar los casos a revision de vertices.

si son poligonos iguales se deja el mas reciente.
si estan en el mismo espacio se deja el mas grande
si se solapan hay dos casos:
    se deja el mas grande
    se ajusta el poligono mas pequeño a los bordes del mas grande.
        (revisar con Qgis, Herramienta "Validate Coverage")

ya se debe de tener la capa CP_2025.shp resultante del proceso de los dos primero codigos de este notebook

'''


import geopandas as gpd
import pandas as pd
import os

from shapely.validation import make_valid
from shapely.geometry import Polygon, MultiPolygon


# ============================================================
# 1. RUTA DEL SHAPEFILE
# ============================================================

ruta_shp = r"C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\TEMP\CP_2026.shp"


# ============================================================
# 2. ARCHIVOS DE SALIDA
# ============================================================

carpeta_salida = os.path.dirname(ruta_shp)

ruta_shp_limpio = os.path.join(
    carpeta_salida,
    "poligonos_limpios.shp"
)

ruta_excel = os.path.join(
    carpeta_salida,
    "resultado_poligonos.xlsx"
)


# ============================================================
# 3. PARAMETRO PRINCIPAL
# ============================================================
#
# Si una superposición parcial representa <= 10 % del
# polígono MENOR, NO se elimina el polígono.
#
# Se elimina únicamente la zona de intersección del menor.
#
# Puedes probar:
#
# 5  -> más estricto
# 10 -> recomendado inicialmente
# 15 -> más tolerante
#
# ============================================================

PORCENTAJE_MAX_AJUSTE = 10.0


# ============================================================
# 4. FUNCION PARA VALIDAR GEOMETRIA
# ============================================================

def validar_geometria(geom):

    if geom is None:
        return None

    if geom.is_empty:
        return geom

    try:

        return make_valid(geom)

    except Exception:

        return geom.buffer(0)


# ============================================================
# 5. LEER SHAPEFILE
# ============================================================

print("\n")
print("==============================================")
print("LECTURA DEL SHAPEFILE")
print("==============================================")

gdf = gpd.read_file(
    ruta_shp
)

print(
    f"Polígonos encontrados: {len(gdf)}"
)


# ============================================================
# 6. VALIDAR CODSIA
# ============================================================

if "CODSIA" not in gdf.columns:

    raise ValueError(
        "El Shapefile no contiene la columna 'CODSIA'."
    )


# ============================================================
# 7. VALIDAR CRS
# ============================================================

if gdf.crs is None:

    raise ValueError(
        "El Shapefile no tiene sistema de coordenadas."
    )


print(
    f"CRS original: {gdf.crs}"
)


# ============================================================
# 8. REPROYECTAR A EPSG:21896
# ============================================================

if gdf.crs.to_epsg() != 21896:

    print(
        "\nReproyectando a EPSG:21896..."
    )

    gdf = gdf.to_crs(
        epsg=21896
    )


print(
    f"CRS utilizado: {gdf.crs}"
)


# ============================================================
# 9. VALIDAR GEOMETRIAS
# ============================================================

print(
    "\nValidando geometrías..."
)

gdf["geometry"] = (
    gdf["geometry"]
    .apply(validar_geometria)
)


# Eliminar geometrías vacías

gdf = gdf[
    gdf.geometry.notna()
    &
    ~gdf.geometry.is_empty
].copy()


gdf = gdf.reset_index(
    drop=True
)


# ============================================================
# 10. CALCULAR AREA ORIGINAL
# ============================================================

gdf["AREA_ORIGINAL_M2"] = (
    gdf.geometry.area
)


# ============================================================
# 11. COLUMNAS DE CONTROL
# ============================================================

gdf["ESTADO"] = "CONSERVADO"

gdf["CODSIA_CONSERVADO"] = None

gdf["AREA_INTERSECCION_M2"] = 0.0

gdf["PORCENTAJE_INTERSECCION"] = 0.0

gdf["MOTIVO"] = "Sin superposición"


# ============================================================
# 12. INDICE ESPACIAL
# ============================================================

sindex = gdf.sindex


# ============================================================
# 13. BUSCAR PAREJAS CON INTERSECCION
# ============================================================

print("\n")
print("==============================================")
print("BUSCANDO SUPERPOSICIONES")
print("==============================================")


parejas = []


for i in range(
    len(gdf)
):

    geom_i = gdf.geometry.iloc[i]


    candidatos = sindex.query(
        geom_i,
        predicate="intersects"
    )


    for j in candidatos:

        # No compararse consigo mismo
        if j <= i:

            continue


        geom_j = gdf.geometry.iloc[j]


        try:

            interseccion = (
                geom_i.intersection(
                    geom_j
                )
            )

        except Exception:

            geom_i = validar_geometria(
                geom_i
            )

            geom_j = validar_geometria(
                geom_j
            )

            interseccion = (
                geom_i.intersection(
                    geom_j
                )
            )


        # ----------------------------------------------------
        # Si no hay interseccion
        # ----------------------------------------------------

        if interseccion.is_empty:

            continue


        area_interseccion = (
            interseccion.area
        )


        # ----------------------------------------------------
        # SOLO BORDE O VERTICE
        #
        # Area = 0
        #
        # No se considera conflicto.
        # ----------------------------------------------------

        if area_interseccion <= 0:

            continue


        parejas.append(
            (
                i,
                j
            )
        )


print(
    f"Superposiciones con área: {len(parejas)}"
)


# ============================================================
# 14. CONTADORES
# ============================================================

contador_ajustados = 0

contador_eliminados = 0

contador_borde = 0


# ============================================================
# 15. PROCESAR PAREJAS
# ============================================================

print("\n")
print("==============================================")
print("PROCESANDO POLIGONOS")
print("==============================================")


for numero, (
    i,
    j
) in enumerate(
    parejas,
    start=1
):


    # ========================================================
    # SI ALGUNO YA FUE ELIMINADO
    # ========================================================

    if (
        gdf.loc[
            i,
            "ESTADO"
        ]
        ==
        "ELIMINADO"
    ):

        continue


    if (
        gdf.loc[
            j,
            "ESTADO"
        ]
        ==
        "ELIMINADO"
    ):

        continue


    # ========================================================
    # OBTENER GEOMETRIAS ACTUALES
    # ========================================================

    geom_i = validar_geometria(
        gdf.loc[
            i,
            "geometry"
        ]
    )

    geom_j = validar_geometria(
        gdf.loc[
            j,
            "geometry"
        ]
    )


    # ========================================================
    # CALCULAR INTERSECCION
    # ========================================================

    try:

        interseccion = (
            geom_i.intersection(
                geom_j
            )
        )

    except Exception:

        continue


    # ========================================================
    # NO HAY INTERSECCION
    # ========================================================

    if interseccion.is_empty:

        continue


    area_interseccion = (
        interseccion.area
    )


    # ========================================================
    # SOLO BORDE
    # ========================================================

    if area_interseccion <= 0:

        contador_borde += 1

        continue


    # ========================================================
    # AREAS
    # ========================================================

    area_i = geom_i.area

    area_j = geom_j.area


    # ========================================================
    # DETERMINAR MAYOR Y MENOR
    # ========================================================

    if area_i >= area_j:

        indice_mayor = i

        indice_menor = j

        area_mayor = area_i

        area_menor = area_j

    else:

        indice_mayor = j

        indice_menor = i

        area_mayor = area_j

        area_menor = area_i


    # ========================================================
    # PORCENTAJE DE INTERSECCION SOBRE EL MENOR
    # ========================================================

    if area_menor > 0:

        porcentaje_interseccion = (
            area_interseccion /
            area_menor
        ) * 100

    else:

        porcentaje_interseccion = 100


    # ========================================================
    # GUARDAR INFORMACION
    # ========================================================

    gdf.loc[
        i,
        "AREA_INTERSECCION_M2"
    ] = area_interseccion


    gdf.loc[
        j,
        "AREA_INTERSECCION_M2"
    ] = area_interseccion


    gdf.loc[
        i,
        "PORCENTAJE_INTERSECCION"
    ] = porcentaje_interseccion


    gdf.loc[
        j,
        "PORCENTAJE_INTERSECCION"
    ] = porcentaje_interseccion


    # ========================================================
    # 16. DETECTAR SI UNO ESTA DENTRO DEL OTRO
    # ========================================================
    #
    # Este caso NO se trata como una pequeña intersección.
    #
    # Si B esta completamente dentro de A:
    #
    # A -> conservar
    # B -> eliminar
    #
    # Esto mantiene la lógica que ya te funcionaba.
    # ========================================================

    try:

        menor_dentro_mayor = (
            geom_mayor.covers(
                geom_menor
            )
        )

    except Exception:

        menor_dentro_mayor = False


    # ========================================================
    # CASO 1
    # UNO ESTA COMPLETAMENTE DENTRO DEL OTRO
    # ========================================================

    if menor_dentro_mayor:


        codsia_mayor = (
            gdf.loc[
                indice_mayor,
                "CODSIA"
            ]
        )


        codsia_menor = (
            gdf.loc[
                indice_menor,
                "CODSIA"
            ]
        )


        gdf.loc[
            indice_menor,
            "ESTADO"
        ] = "ELIMINADO"


        gdf.loc[
            indice_menor,
            "CODSIA_CONSERVADO"
        ] = codsia_mayor


        gdf.loc[
            indice_menor,
            "MOTIVO"
        ] = (
            "Polígono completamente dentro "
            "de otro; se conserva el mayor"
        )


        contador_eliminados += 1


        print(
            f"[{numero}] "
            f"DENTRO -> ELIMINADO | "
            f"{codsia_menor} | "
            f"Mayor: {codsia_mayor}"
        )


        continue


    # ========================================================
    # CASO 2
    # SUPERPOSICION PARCIAL PEQUEÑA
    # ========================================================
    #
    # ESTA ES LA PARTE IMPORTANTE.
    #
    # Ya NO usamos:
    #
    #     area < 5 m2
    #
    # sino:
    #
    #     % interseccion del menor
    #
    # Si <= 10%, solamente eliminamos esa zona del menor.
    # ========================================================

    if (
        porcentaje_interseccion
        <=
        PORCENTAJE_MAX_AJUSTE
    ):


        geom_mayor = (
            gdf.loc[
                indice_mayor,
                "geometry"
            ]
        )


        geom_menor = (
            gdf.loc[
                indice_menor,
                "geometry"
            ]
        )


        # ----------------------------------------------------
        # QUITAR LA INTERSECCION SOLAMENTE DEL MENOR
        # ----------------------------------------------------

        try:

            geom_menor_nueva = (
                geom_menor.difference(
                    geom_mayor
                )
            )

        except Exception:

            geom_menor = (
                validar_geometria(
                    geom_menor
                )
            )

            geom_mayor = (
                validar_geometria(
                    geom_mayor
                )
            )

            geom_menor_nueva = (
                geom_menor.difference(
                    geom_mayor
                )
            )


        # ----------------------------------------------------
        # VALIDAR RESULTADO
        # ----------------------------------------------------

        if (
            geom_menor_nueva is not None
            and
            not geom_menor_nueva.is_empty
            and
            geom_menor_nueva.area > 0
        ):


            geom_menor_nueva = (
                validar_geometria(
                    geom_menor_nueva
                )
            )


            # ------------------------------------------------
            # IMPORTANTE:
            #
            # SOLO SE MODIFICA EL MENOR.
            #
            # EL MAYOR NO SE TOCA.
            # ------------------------------------------------

            gdf.loc[
                indice_menor,
                "geometry"
            ] = geom_menor_nueva


            gdf.loc[
                indice_menor,
                "MOTIVO"
            ] = (
                "Superposición parcial pequeña; "
                "se eliminó únicamente la intersección "
                "del polígono menor"
            )


            gdf.loc[
                indice_menor,
                "CODSIA_CONSERVADO"
            ] = gdf.loc[
                indice_mayor,
                "CODSIA"
            ]


            contador_ajustados += 1


            print(
                f"[{numero}] "
                f"AJUSTADO | "
                f"Menor: "
                f"{gdf.loc[indice_menor, 'CODSIA']} | "
                f"Mayor: "
                f"{gdf.loc[indice_mayor, 'CODSIA']} | "
                f"Intersección: "
                f"{area_interseccion:.2f} m² | "
                f"Porcentaje: "
                f"{porcentaje_interseccion:.3f}%"
            )


        else:

            print(
                f"[{numero}] "
                f"No se pudo ajustar: "
                f"{gdf.loc[indice_menor, 'CODSIA']}"
            )


    # ========================================================
    # CASO 3
    # SUPERPOSICION PARCIAL GRANDE
    # ========================================================
    #
    # Si no esta completamente dentro y la intersección
    # representa mas del porcentaje permitido:
    #
    # -> conservar el mayor
    # -> eliminar el menor
    # ========================================================

    else:


        codsia_mayor = (
            gdf.loc[
                indice_mayor,
                "CODSIA"
            ]
        )


        codsia_menor = (
            gdf.loc[
                indice_menor,
                "CODSIA"
            ]
        )


        gdf.loc[
            indice_menor,
            "ESTADO"
        ] = "ELIMINADO"


        gdf.loc[
            indice_menor,
            "CODSIA_CONSERVADO"
        ] = codsia_mayor


        gdf.loc[
            indice_menor,
            "MOTIVO"
        ] = (
            "Superposición parcial significativa; "
            "se conserva el polígono de mayor área"
        )


        contador_eliminados += 1


        print(
            f"[{numero}] "
            f"ELIMINADO | "
            f"Menor: {codsia_menor} | "
            f"Mayor: {codsia_mayor} | "
            f"Intersección: "
            f"{area_interseccion:.2f} m² | "
            f"Porcentaje: "
            f"{porcentaje_interseccion:.3f}%"
        )


# ============================================================
# 17. AREA FINAL
# ============================================================

gdf["AREA_FINAL_M2"] = (
    gdf.geometry.area
)


# ============================================================
# 18. CREAR EXCEL
# ============================================================

resultado_excel = gdf[
    [
        "CODSIA",
        "AREA_ORIGINAL_M2",
        "AREA_FINAL_M2",
        "ESTADO",
        "CODSIA_CONSERVADO",
        "AREA_INTERSECCION_M2",
        "PORCENTAJE_INTERSECCION",
        "MOTIVO"
    ]
].copy()


# ============================================================
# 19. TIPO DE RESULTADO
# ============================================================

def determinar_tipo(row):

    if row["ESTADO"] == "ELIMINADO":

        return "ELIMINADO"

    if "Superposición" in str(
        row["MOTIVO"]
    ):

        return "AJUSTADO"

    return "CONSERVADO"


resultado_excel[
    "TIPO_RESULTADO"
] = resultado_excel.apply(
    determinar_tipo,
    axis=1
)


# ============================================================
# 20. GUARDAR EXCEL
# ============================================================

resultado_excel.to_excel(
    ruta_excel,
    index=False
)


# ============================================================
# 21. CREAR SHAPEFILE FINAL
# ============================================================
#
# IMPORTANTE:
#
# Los AJUSTADOS permanecen.
#
# Solamente salen los ELIMINADOS.
# ============================================================

gdf_limpio = gdf[
    gdf["ESTADO"] != "ELIMINADO"
].copy()


# ============================================================
# 22. ELIMINAR COLUMNAS AUXILIARES
# ============================================================

columnas_auxiliares = [

    "AREA_ORIGINAL_M2",

    "AREA_FINAL_M2",

    "ESTADO",

    "CODSIA_CONSERVADO",

    "AREA_INTERSECCION_M2",

    "PORCENTAJE_INTERSECCION",

    "MOTIVO"
]


gdf_limpio = gdf_limpio.drop(
    columns=[
        columna
        for columna in columnas_auxiliares
        if columna in gdf_limpio.columns
    ]
)


# ============================================================
# 23. GUARDAR SHAPEFILE
# ============================================================

gdf_limpio.to_file(
    ruta_shp_limpio,
    driver="ESRI Shapefile",
    encoding="utf-8"
)


# ============================================================
# 24. RESUMEN FINAL
# ============================================================

print("\n")
print("==============================================")
print("             PROCESO TERMINADO")
print("==============================================")


print(
    f"Polígonos iniciales   : {len(gdf)}"
)

print(
    f"Polígonos ajustados   : {contador_ajustados}"
)

print(
    f"Polígonos eliminados  : {contador_eliminados}"
)

print(
    f"Superposiciones       : {len(parejas)}"
)


print("\n")
print("----------------------------------------------")
print("SHAPEFILE LIMPIO")
print("----------------------------------------------")

print(
    ruta_shp_limpio
)


print("\n")
print("----------------------------------------------")
print("EXCEL")
print("----------------------------------------------")

print(
    ruta_excel
)


print("\n")
print("==============================================")



LECTURA DEL SHAPEFILE
Polígonos encontrados: 2640
CRS original: EPSG:21896
CRS utilizado: EPSG:21896

Validando geometrías...


BUSCANDO SUPERPOSICIONES
Superposiciones con área: 74


PROCESANDO POLIGONOS
[1] ELIMINADO | Menor: 010001015F | Mayor: 010001015A | Intersección: 2523.63 m² | Porcentaje: 100.000%
[2] ELIMINADO | Menor: 010001015F | Mayor: 010001015A | Intersección: 6304.42 m² | Porcentaje: 100.000%
[3] ELIMINADO | Menor: 010226006F | Mayor: 010226006E | Intersección: 276715.52 m² | Porcentaje: 100.000%
[4] ELIMINADO | Menor: 010224004 | Mayor: 010224004C | Intersección: 7291.19 m² | Porcentaje: 100.000%
[5] ELIMINADO | Menor: 010103009D | Mayor: 010103009A | Intersección: 188507.50 m² | Porcentaje: 100.000%
[6] ELIMINADO | Menor: 010103009D | Mayor: 010103009A | Intersección: 5055.85 m² | Porcentaje: 100.000%
[7] ELIMINADO | Menor: 012259004 | Mayor: 012259004A | Intersección: 295698.57 m² | Porcentaje: 100.000%
[8] ELIMINADO | Menor: 011595001A | Mayor: 011595001 | Inters